# ReviewAgent — Knowledge Graph Visualization

This notebook demonstrates Stage 2's Knowledge Graph construction:
1. Build a KG from sample reviews using the Stage 2 pipeline
2. Visualize the graph structure (aspect nodes, entity nodes, edges)
3. Show clustering and priority ranking

In [ ]:
import sys
sys.path.insert(0, "..")

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from datetime import datetime

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Create Sample Reviews

In [ ]:
from src.common.schemas import ReviewObject, AspectSentiment, ExtractedEntities

# Simulated Stage 1 output — reviews with labels, aspects, and entities
reviews = [
    ReviewObject(
        review_id="r1", text="App crashes every time I try to login on my Samsung", rating=1,
        app_id="app1", timestamp=datetime(2026, 3, 1),
        labels=["bug_report"], aspects=[AspectSentiment(aspect="login", sentiment="negative", intensity=0.9)],
        entities=ExtractedEntities(devices=["Samsung Galaxy S24"], os_versions=["Android 14"]),
    ),
    ReviewObject(
        review_id="r2", text="Login screen freezes then app closes", rating=1,
        app_id="app1", timestamp=datetime(2026, 3, 2),
        labels=["bug_report"], aspects=[AspectSentiment(aspect="login", sentiment="negative", intensity=0.85)],
        entities=ExtractedEntities(devices=["Pixel 8"], os_versions=["Android 14"]),
    ),
    ReviewObject(
        review_id="r3", text="Cannot log in since the last update", rating=2,
        app_id="app1", timestamp=datetime(2026, 3, 3),
        labels=["bug_report"], aspects=[AspectSentiment(aspect="login", sentiment="negative", intensity=0.8)],
        entities=ExtractedEntities(app_versions=["v3.2"]),
    ),
    ReviewObject(
        review_id="r4", text="Please add dark mode, it's so bright at night", rating=3,
        app_id="app1", timestamp=datetime(2026, 3, 1),
        labels=["feature_request"], aspects=[AspectSentiment(aspect="dark_mode", sentiment="negative", intensity=0.6)],
        entities=ExtractedEntities(),
    ),
    ReviewObject(
        review_id="r5", text="Would love a dark theme option for the app", rating=3,
        app_id="app1", timestamp=datetime(2026, 3, 5),
        labels=["feature_request"], aspects=[AspectSentiment(aspect="dark_mode", sentiment="negative", intensity=0.5)],
        entities=ExtractedEntities(),
    ),
    ReviewObject(
        review_id="r6", text="Battery drain is terrible with this app running", rating=1,
        app_id="app1", timestamp=datetime(2026, 3, 4),
        labels=["performance"], aspects=[AspectSentiment(aspect="battery", sentiment="negative", intensity=0.95)],
        entities=ExtractedEntities(devices=["Samsung Galaxy S24"]),
    ),
    ReviewObject(
        review_id="r7", text="App uses too much battery in background", rating=2,
        app_id="app1", timestamp=datetime(2026, 3, 6),
        labels=["performance"], aspects=[AspectSentiment(aspect="battery", sentiment="negative", intensity=0.7)],
        entities=ExtractedEntities(),
    ),
    ReviewObject(
        review_id="r8", text="Love this app! Best photo editor ever!", rating=5,
        app_id="app1", timestamp=datetime(2026, 3, 2),
        labels=["praise"], aspects=[AspectSentiment(aspect="overall", sentiment="positive", intensity=0.9)],
        entities=ExtractedEntities(),
    ),
]

print(f"Created {len(reviews)} sample reviews across {len(set(r.labels[0] for r in reviews))} categories")

## 2. Build Knowledge Graph

In [ ]:
from src.stage2.kg_builder import KnowledgeGraphBuilder

kg_builder = KnowledgeGraphBuilder()
G = kg_builder.build(reviews)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"\nNode types:")
for ntype in set(nx.get_node_attributes(G, 'type').values()):
    count = sum(1 for _, d in G.nodes(data=True) if d.get('type') == ntype)
    print(f"  {ntype}: {count}")
print(f"\nEdge types:")
for etype in set(nx.get_edge_attributes(G, 'relation').values()):
    count = sum(1 for _, _, d in G.edges(data=True) if d.get('relation') == etype)
    print(f"  {etype}: {count}")

## 3. Visualize the Knowledge Graph

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

# Color by node type
type_colors = {
    "review": "#4ECDC4",
    "aspect": "#FF6B6B",
    "entity": "#45B7D1",
    "device": "#96CEB4",
    "os_version": "#FFEAA7",
    "app_version": "#DDA0DD",
}

node_colors = []
node_sizes = []
for n, d in G.nodes(data=True):
    ntype = d.get("type", "unknown")
    node_colors.append(type_colors.get(ntype, "#CCCCCC"))
    if ntype == "aspect":
        node_sizes.append(800)
    elif ntype == "review":
        node_sizes.append(400)
    else:
        node_sizes.append(300)

# Edge colors by relation
edge_colors = []
for u, v, d in G.edges(data=True):
    rel = d.get("relation", "")
    if "sentiment" in rel:
        edge_colors.append("#FF6B6B" if "negative" in str(d.get("sentiment", "")) else "#4ECDC4")
    elif "has_entity" in rel or "mentioned" in rel:
        edge_colors.append("#45B7D1")
    else:
        edge_colors.append("#CCCCCC")

pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

nx.draw_networkx_edges(G, pos, edge_color=edge_colors, alpha=0.4, width=1.5, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8, ax=ax)

# Labels — shorten review texts
label_map = {}
for n, d in G.nodes(data=True):
    if d.get("type") == "review":
        label_map[n] = n  # Just show review ID
    else:
        label_map[n] = str(n)[:20]

nx.draw_networkx_labels(G, pos, label_map, font_size=8, ax=ax)

# Legend
patches = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=patches, loc='upper left', fontsize=10)
ax.set_title("ReviewAgent Knowledge Graph — Sample Reviews", fontsize=14)
ax.axis('off')

plt.tight_layout()
plt.savefig("../notebooks/fig_knowledge_graph.png", dpi=150, bbox_inches='tight')
plt.show()

## 4. Graph Centrality (Priority Ranking)

In [ ]:
# PageRank and betweenness centrality
pagerank = nx.pagerank(G)
betweenness = nx.betweenness_centrality(G)
degree = dict(G.degree())

# Show top nodes by centrality
centrality_df = pd.DataFrame({
    "Node": list(pagerank.keys()),
    "PageRank": list(pagerank.values()),
    "Betweenness": [betweenness[n] for n in pagerank.keys()],
    "Degree": [degree[n] for n in pagerank.keys()],
    "Type": [G.nodes[n].get("type", "?") for n in pagerank.keys()],
})

import pandas as pd

print("Top 10 nodes by PageRank (highest priority):")
centrality_df.sort_values("PageRank", ascending=False).head(10).to_string(index=False)
centrality_df.sort_values("PageRank", ascending=False).head(10)

In [ ]:
# Centrality visualization — size by PageRank
fig, ax = plt.subplots(figsize=(14, 10))

pr_sizes = [pagerank[n] * 10000 + 100 for n in G.nodes()]

nx.draw_networkx_edges(G, pos, edge_color="#CCCCCC", alpha=0.3, width=1, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=pr_sizes, alpha=0.8, ax=ax, edgecolors="black", linewidths=0.5)
nx.draw_networkx_labels(G, pos, label_map, font_size=8, ax=ax)

ax.set_title("Knowledge Graph — Node Size by PageRank Centrality", fontsize=14)
ax.axis('off')

plt.tight_layout()
plt.savefig("../notebooks/fig_kg_centrality.png", dpi=150, bbox_inches='tight')
plt.show()